# Preprocess RAL-CLIP Local Tokens - FF++

Notebook này extract và cache CLIP **global embedding + local patch tokens** cho **FaceForensics++** để dùng với RAL-CLIP. Output lưu theo shard `.pt`, phù hợp cho FF++ clean hoặc FF++ corruption.


## Kaggle Setup

In [ ]:
# Chạy cell này trên Kaggle nếu repo chưa có trong /kaggle/working.
# !git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
# %cd /kaggle/working/training-free-tta-for-deepfake-detection
# !pip install -q -e . --no-deps
# !pip install -q open_clip_torch

## Imports

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/training-free-tta-for-deepfake-detection')]
    for root in candidates:
        if (root / 'code' / 'deepfake_tta').exists():
            return root.resolve(), (root / 'code').resolve()
        if (root / 'deepfake_tta').exists():
            return root.resolve(), root.resolve()
    raise FileNotFoundError('Cannot find repo root containing deepfake_tta')

repo_root, code_root = find_repo_root()
sys.path.insert(0, str(code_root))

from training.ffpp_split_utils import prepare_ffpp_split_dataframe

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo_root:', repo_root)
print('code_root:', code_root)
print('device:', DEVICE)

## Config

Đổi các path trong cell này theo dataset đang mount. Với test corruption, set `REPLACEMENT_ROOT` tới folder level chứa cấu trúc dataset tương ứng.

In [ ]:
CSV_PATH = Path('/kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv')
DEEPFAKEBENCH_ROOT = Path('/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench')

# Dataset cần preprocess.
DATASET_NAME = 'FaceForensics++'  # FF++
SPLIT_NAME = 'test'                # 'train'/'val'/'test'
SPLIT_ROOT = None                  # Folder chứa train.json/val.json/test.json nếu cần.

# Clean: để None. Corruption: trỏ tới root đã xử lý, ví dụ .../gaussian_blur/level_2/FaceForensics++ hoặc .../level_2.
REPLACEMENT_ROOT = None

# Nếu chỉ muốn source real memory cho RAL-CLIP thì bật ONLY_REAL=True.
ONLY_REAL = False
MAX_SAMPLES = None                 # Ví dụ 2000 để test nhanh, None để chạy full.

CLIP_MODEL = 'ViT-L-14'
PRETRAINED = 'openai'
LAYERS = [-6]                      # Bắt đầu bằng 1 layer để nhẹ. Có thể thử [-8, -6, -4].
BATCH_SIZE = 16
NUM_WORKERS = 2
USE_AMP = True
SAVE_DTYPE = 'float16'             # 'float16' tiết kiệm disk; 'float32' nếu muốn chính xác tối đa.

# Số ảnh mỗi shard. Local tokens rất nặng, nên 256-1024 là hợp lý tùy disk/RAM.
SHARD_SIZE = 512
SEED = 42

OUTPUT_DIR = Path('/kaggle/working/ral_clip_local_tokens')
OUTPUT_PREFIX = 'ffpp_test_clean_vitl14_layer-6'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Prepare DataFrame

In [ ]:
def find_dataset_root(root, dataset_name):
    root = Path(root)
    candidates = [root / dataset_name, root]
    if dataset_name == 'Celeb-DF-v1':
        anchors = ('Celeb-real', 'YouTube-real', 'Celeb-synthesis')
        for candidate in candidates:
            if any((candidate / anchor).exists() for anchor in anchors):
                return candidate
    if dataset_name == 'FaceForensics++':
        anchors = ('original_sequences', 'manipulated_sequences')
        for candidate in candidates:
            if any((candidate / anchor).exists() for anchor in anchors):
                return candidate
    return root

def prepare_generic_dataframe(csv_path, dataset_name, deepfakebench_root, replacement_root=None):
    df = pd.read_csv(csv_path)
    df = df[df['datasetname'].eq(dataset_name)].copy()
    if df.empty:
        raise ValueError(f'No rows found for datasetname={dataset_name!r}')
    df['label_num'] = df['label'].map({'REAL': 0, 'FAKE': 1}).astype(int)
    df['imagepath_fixed'] = df['imagepath'].astype(str).str.replace('../input/deepfakebench', str(deepfakebench_root), regex=False)

    if replacement_root is not None:
        source_root = Path(deepfakebench_root) / dataset_name
        target_root = find_dataset_root(replacement_root, dataset_name)
        df['imagepath_fixed'] = df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(target_root), regex=False)
        print('replacement root:', target_root)

    exists = df['imagepath_fixed'].map(lambda p: Path(p).exists())
    missing = int((~exists).sum())
    if missing:
        print(f'dropping missing files: {missing}/{len(df)}')
        print(df.loc[~exists, ['imagepath', 'imagepath_fixed']].head(10).to_string(index=False))
        df = df[exists].copy()
    return df.reset_index(drop=True)

def sample_dataframe(df, max_samples, seed):
    if max_samples is None or len(df) <= max_samples:
        return df.reset_index(drop=True)
    if set(df['label_num'].unique()) == {0, 1}:
        per_class = max_samples // 2
        parts = []
        for label in [0, 1]:
            sub = df[df['label_num'].eq(label)]
            parts.append(sub.sample(n=min(per_class, len(sub)), random_state=seed))
        return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df.sample(n=max_samples, random_state=seed).reset_index(drop=True)

if DATASET_NAME == 'FaceForensics++' and SPLIT_NAME:
    df = prepare_ffpp_split_dataframe(
        CSV_PATH,
        DEEPFAKEBENCH_ROOT,
        split_name=SPLIT_NAME,
        split_root=SPLIT_ROOT,
    )
    if REPLACEMENT_ROOT is not None:
        source_root = Path(DEEPFAKEBENCH_ROOT) / 'FaceForensics++'
        target_root = find_dataset_root(REPLACEMENT_ROOT, 'FaceForensics++')
        df['imagepath_fixed'] = df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(target_root), regex=False)
        print('replacement root:', target_root)
else:
    df = prepare_generic_dataframe(CSV_PATH, DATASET_NAME, DEEPFAKEBENCH_ROOT, replacement_root=REPLACEMENT_ROOT)

if ONLY_REAL:
    df = df[df['label_num'].eq(0)].reset_index(drop=True)

df = sample_dataframe(df, MAX_SAMPLES, SEED)

print('rows:', len(df))
print(df['label_num'].value_counts().rename(index={0: 'REAL', 1: 'FAKE'}))
display(df[['label', 'label_num', 'imagepath_fixed']].head())

## CLIP Local Token Extractor

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, dataframe, preprocess):
        self.df = dataframe.reset_index(drop=True)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['imagepath_fixed']).convert('RGB')
        image = self.preprocess(image)
        return image, int(row['label_num']), row['imagepath_fixed']

class OpenClipLocalExtractor:
    def __init__(self, model, layers):
        self.model = model.eval()
        self.layers = list(layers)
        self.captures = {}
        self.handles = []
        blocks = self._visual_blocks()
        n_blocks = len(blocks)
        self.layer_indices = [layer if layer >= 0 else n_blocks + layer for layer in self.layers]
        for idx in self.layer_indices:
            if idx < 0 or idx >= n_blocks:
                raise IndexError(f'Layer {idx} out of range for {n_blocks} blocks')
            self.handles.append(blocks[idx].register_forward_hook(self._make_hook(idx)))
        print('hooked visual layer indices:', self.layer_indices)
    def _visual_blocks(self):
        visual = self.model.visual
        if hasattr(visual, 'transformer') and hasattr(visual.transformer, 'resblocks'):
            return visual.transformer.resblocks
        if hasattr(visual, 'trunk') and hasattr(visual.trunk, 'blocks'):
            return visual.trunk.blocks
        raise TypeError('Cannot find visual transformer blocks. Use OpenCLIP ViT model, e.g. ViT-L-14/openai.')
    def _make_hook(self, idx):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            self.captures[idx] = x.detach()
        return hook
    def close(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []
    @torch.inference_mode()
    def __call__(self, images):
        self.captures = {}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(USE_AMP and images.device.type == 'cuda')):
            global_features = self.model.encode_image(images)
        global_features = F.normalize(global_features.float(), dim=-1)

        locals_by_layer = {}
        batch_size = images.shape[0]
        for raw_layer, idx in zip(self.layers, self.layer_indices):
            tokens = self.captures[idx].float()
            if tokens.shape[0] == batch_size:
                tokens = tokens
            elif tokens.shape[1] == batch_size:
                tokens = tokens.permute(1, 0, 2).contiguous()
            else:
                raise ValueError(f'Cannot infer batch dim from token shape {tuple(tokens.shape)}')
            patch_tokens = F.normalize(tokens[:, 1:, :], dim=-1)
            locals_by_layer[f'layer_{raw_layer}'] = patch_tokens.cpu()
        return global_features.cpu(), locals_by_layer

def cast_for_save(tensor):
    if SAVE_DTYPE == 'float16':
        return tensor.half()
    if SAVE_DTYPE == 'float32':
        return tensor.float()
    raise ValueError(f'Unknown SAVE_DTYPE={SAVE_DTYPE}')

## Run Extraction

In [ ]:
import open_clip

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL,
    pretrained=PRETRAINED,
    device=DEVICE,
)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad_(False)

extractor = OpenClipLocalExtractor(clip_model, LAYERS)

loader = DataLoader(
    ImagePathDataset(df, preprocess),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

manifest = []
shard = {'global': [], 'local': {f'layer_{layer}': [] for layer in LAYERS}, 'labels': [], 'paths': []}
shard_idx = 0
seen = 0

def flush_shard(force=False):
    global shard_idx, shard
    if not shard['labels']:
        return
    n = sum(len(x) for x in shard['labels'])
    if not force and n < SHARD_SIZE:
        return

    global_tensor = cast_for_save(torch.cat(shard['global']).contiguous())
    local_tensors = {key: cast_for_save(torch.cat(parts).contiguous()) for key, parts in shard['local'].items()}
    labels_tensor = torch.cat(shard['labels']).long().contiguous()
    paths = list(shard['paths'])
    out_path = OUTPUT_DIR / f'{OUTPUT_PREFIX}_shard{shard_idx:04d}.pt'
    payload = {
        'global': global_tensor,
        'local': local_tensors,
        'labels': labels_tensor,
        'paths': paths,
        'metadata': {
            'dataset_name': DATASET_NAME,
            'split_name': SPLIT_NAME,
            'replacement_root': None if REPLACEMENT_ROOT is None else str(REPLACEMENT_ROOT),
            'clip_model': f'{CLIP_MODEL}/{PRETRAINED}',
            'layers': list(LAYERS),
            'save_dtype': SAVE_DTYPE,
            'only_real': bool(ONLY_REAL),
        },
    }
    torch.save(payload, out_path)
    row = {
        'shard': shard_idx,
        'path': str(out_path),
        'samples': int(labels_tensor.numel()),
        'global_shape': tuple(global_tensor.shape),
        'label_counts': torch.bincount(labels_tensor, minlength=2).tolist(),
    }
    for key, value in local_tensors.items():
        row[f'{key}_shape'] = tuple(value.shape)
    manifest.append(row)
    print('saved:', out_path, '| samples:', row['samples'], '| counts:', row['label_counts'])

    shard_idx += 1
    shard = {'global': [], 'local': {f'layer_{layer}': [] for layer in LAYERS}, 'labels': [], 'paths': []}

for images, labels, paths in tqdm(loader):
    images = images.to(DEVICE, non_blocking=True)
    global_features, local_features = extractor(images)
    shard['global'].append(global_features)
    for key, value in local_features.items():
        shard['local'][key].append(value)
    shard['labels'].append(labels.cpu())
    shard['paths'].extend(paths)
    seen += len(labels)
    if sum(len(x) for x in shard['labels']) >= SHARD_SIZE:
        flush_shard(force=True)

flush_shard(force=True)
extractor.close()

manifest_df = pd.DataFrame(manifest)
manifest_path = OUTPUT_DIR / f'{OUTPUT_PREFIX}_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)
print('saved manifest:', manifest_path)
display(manifest_df)

## Inspect One Shard

In [ ]:
first_shard = Path(manifest_df.iloc[0]['path'])
payload = torch.load(first_shard, map_location='cpu')
print('file:', first_shard)
print('global:', payload['global'].shape, payload['global'].dtype)
for key, value in payload['local'].items():
    print(key, value.shape, value.dtype)
print('labels:', payload['labels'].shape, torch.bincount(payload['labels'], minlength=2).tolist())
print('metadata:', payload['metadata'])
payload['paths'][:3]